In [ ]:
!unzip /content/drive/MyDrive/AI_HUB_DAMAGE_DATASET.zip

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401472_sc-155261.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401525_sc-151864.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401601_sc-207688.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401605_sc-207688.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401616_as-7439434.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401645_sc-1023739.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401683_as-0074373.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401698_sc-134277.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401721_sc-215212.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401736_as-0048988.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401751_as-0047594.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401854_as-0059459.txt  
  inflating: AI_HUB_DAMAGE_DATASET/labels/train/0401861_sc-188612.txt  
  inflating: AI_HUB_DA

In [ ]:
!mv /content/AI_HUB_DAMAGE_DATASET /content/drive/MyDrive/.

In [ ]:
import json
import os
import cv2
from tqdm import tqdm

def create_coco_json(image_root, label_root, save_path):
    coco_format = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": 0, "name": "scratch"},
            {"id": 1, "name": "dent"},
            {"id": 2, "name": "crushed"},
            {"id": 3, "name": "separated"}
        ]
    }

    ann_id = 0
    img_list = [f for f in os.listdir(image_root) if f.endswith(('.jpg', '.jpeg', '.png'))]

    for img_id, file_name in enumerate(tqdm(img_list)):
        img_path = os.path.join(image_root, file_name)
        img = cv2.imread(img_path)
        h, w, _ = img.shape

        coco_format["images"].append({
            "id": img_id, "file_name": file_name, "width": w, "height": h
        })

        label_path = os.path.join(label_root, os.path.splitext(file_name)[0] + '.txt')
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    cls_id, cx, cy, bw, bh = map(float, line.split())
                    # YOLO to COCO
                    abs_w, abs_h = bw * w, bh * h
                    abs_x, abs_y = (cx * w) - (abs_w / 2), (cy * h) - (abs_h / 2)

                    coco_format["annotations"].append({
                        "id": ann_id, "image_id": img_id, "category_id": int(cls_id),
                        "bbox": [abs_x, abs_y, abs_w, abs_h],
                        "area": abs_w * abs_h, "iscrowd": 0, "segmentation": []
                    })
                    ann_id += 1

    with open(save_path, 'w') as f:
        json.dump(coco_format, f)
    print(f"Finished: {save_path}")

# 실행 (경로는 본인의 drive 상황에 맞춰 수정)
ROOT = '/content/drive/MyDrive/AI_HUB_DAMAGE_DATASET'
create_coco_json(f'{ROOT}/images/train', f'{ROOT}/labels/train', f'{ROOT}/train.json')
create_coco_json(f'{ROOT}/images/val', f'{ROOT}/labels/val', f'{ROOT}/val.json')

100%|██████████| 10000/10000 [54:24<00:00,  3.06it/s]


Finished: /content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/train.json


100%|██████████| 2000/2000 [12:29<00:00,  2.67it/s]

Finished: /content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/val.json


In [ ]:
!ls -f /content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/images/val/*.jpg | wc -l

2000


In [ ]:
import yaml
import os

# 1. 경로 설정
ROOT_PATH = '/content/drive/MyDrive/AI_HUB_DAMAGE_DATASET'

data_config = {
    'path': ROOT_PATH,         # 최상위 경로
    'train': 'images/train',   # 학습 이미지 경로
    'val': 'images/val',       # 검증 이미지 경로
    'test': '',                # (선택) 테스트 경로
    'nc': 4,                   # 클래스 수 (scratch, dent, crushed, separated)
    'names': ['scratch', 'dent', 'crushed', 'separated']
}

# 2. yaml 파일 저장
with open(os.path.join(ROOT_PATH, 'damage_data.yaml'), 'w') as f:
    yaml.dump(data_config, f)

print(f"✅ 설정 파일 생성 완료: {ROOT_PATH}/damage_data.yaml")

✅ 설정 파일 생성 완료: /content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/damage_data.yaml


In [ ]:
# 가장 첫 번째 라벨 파일 내용을 확인하는 코드
import os
ROOT_PATH = '/content/drive/MyDrive/AI_HUB_DAMAGE_DATASET'

label_sample_path = os.path.join(ROOT_PATH, 'labels/train')
sample_file = os.listdir(label_sample_path)[0]

with open(os.path.join(label_sample_path, sample_file), 'r') as f:
    print(f"라벨 샘플 내용: {f.readline()}")

라벨 샘플 내용: 0 0.710625 0.865833 0.018750 0.031667



In [ ]:
# 1. 라이브러리 설치
!pip install ultralytics

# 2. 학습 시작
from ultralytics import YOLO

# v11x-seg (최상위 성능 세그멘테이션 모델) 로드
model = YOLO('yolov8m.pt')

# 1. 저장할 폴더 경로 설정
# AI_HUB_DAMAGE_DATASET 폴더 안에 'train_results'라는 이름으로 저장됩니다.
SAVE_PATH = '/content/drive/MyDrive/AI_HUB_DAMAGE_DATASET'

# 2. 학습 실행
model.train(
    data='/content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/damage_data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    verbose=True,
    patience=20,
    # 아래 두 인자가 핵심입니다!
    project=SAVE_PATH,      # 최상위 저장 경로 (드라이브 폴더)
    name='car_damage_model_100ep', # 상세 폴더 이름
    exist_ok=True           # 같은 이름의 폴더가 있어도 덮어쓰거나 이어서 저장
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/damage_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x793ed0045220>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
# 1. 라이브러리 설치 (이미 설치했다면 생략 가능)
# !pip install ultralytics

# 2. RT-DETR 학습 시작
from ultralytics import RTDETR

# RT-DETR 모델 로드 (l: large, x: extra large 중 선택 가능)
# rtdetr-l.pt는 성능과 속도의 균형이 가장 좋습니다.
model = RTDETR('rtdetr-l.pt')

# 1. 저장할 폴더 경로 설정
SAVE_PATH = '/content/drive/MyDrive/AI_HUB_DAMAGE_DATASET'

# 2. 학습 실행
model.train(
    data='/content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/damage_data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    verbose=True,
    patience=20,
    project=SAVE_PATH,      # 최상위 저장 경로
    name='car_damage_rtdetr_100ep', # 상세 폴더 이름 (RT-DETR용으로 구분)
    exist_ok=True           # 폴더 중복 허용
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/damage_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=car_damage_rtdetr_100ep, nbs=64, nms=False, opset=None, optimize=False,

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      12.4G      1.073      2.042     0.6801         87        640: 100% ━━━━━━━━━━━━ 625/625 1.4it/s 7:22
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 4.6it/s 13.7s
                   all       2000       6798      0.445      0.131     0.0877     0.0408

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/100      13.2G     0.8219      0.879     0.4777         96        640: 0% ──────────── 0/625  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      13.2G     0.8103      0.892      0.428        101        640: 100% ━━━━━━━━━━━━ 625/625 1.9it/s 5:32
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 5.4it/s 11.7s
                   all       2000       6798       0.22      0.197      0.123     0.0564

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100      13.2G     0.6599       1.01     0.3557         92        640: 0% ──────────── 0/625  0.4s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     48/100      13.2G     0.6154     0.6549     0.2719         95        640: 100% ━━━━━━━━━━━━ 625/625 2.2it/s 4:49
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 5.3it/s 11.9s
                   all       2000       6798      0.419      0.303      0.264      0.119

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     49/100      13.2G     0.6052     0.6564     0.2638         93        640: 100% ━━━━━━━━━━━━ 625/625 2.2it/s 4:49
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 5.3it/s 11.8s
                   all       2000       6798      0.422      0.292      0.265       0.12
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 29, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

49 epochs completed in 4.180 hours.
Optimizer stripped from /content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/car_damage_rtdetr_100ep/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/car_damage_rtdetr_100ep/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/AI_HUB_DAMAGE_DATASET/car_damage_rtdetr_100ep/weights/best.pt...
Ultralytics 8.4.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x793fb875bda0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0